In [9]:
## ✅ update access token every hour
# ✅ Fetches 2000 posts per hashtag using pagination
# ✅ Uses one request per media load (limits = 100 per request)
# ✅ Automatically handles API rate limits
# ✅ Appends data to one JSON file
import requests
import json
import time
import os

# --- CONFIGURATION ---
ACCESS_TOKEN = "EAAVx7qOTMZBcBOZBf8qMLWGZAI8ZAGZCEGOu4Pya02F0tLrzCVlZCOtexsLd3AZBuA1CMZBsDsUSF4kKpwAaRxzaTnTX7aLZA4auZBeOl1l3cn0pBk1xmJ4LDOSQzZAChMZCKV2WOn7lKTz09ZB73TXFoh0bD2QlV1LwRzn9sW9ZCFARZBNtrfuu3AmxlZAA97Mip0iKoD5iocuvMpAdCY9BlNwkorx5qCWsv4QyMfGYwqEZD"
USER_ID = "17841468142856047"  # Replace with your Instagram Business Account ID
POSTS_PER_HASHTAG = 10000 # Number of posts to fetch per hashtag
BATCH_SIZE = 50  # Max limit per request

JSON_FILENAME = "beauty2.json"

# Get multiple hashtags from user
hashtags = input("Enter hashtags separated by commas (without #): ").strip().split(',')

# --- HELPER FUNCTIONS ---
def get_hashtag_id(hashtag_name, user_id, access_token):
    """Fetch the Instagram hashtag ID for a given hashtag name."""
    url = f"https://graph.facebook.com/v22.0/ig_hashtag_search?user_id={user_id}&q={hashtag_name.strip()}&access_token={access_token}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data["data"][0]["id"] if "data" in data and data["data"] else None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching hashtag ID for #{hashtag_name}: {e}")
        return None

def fetch_posts(hashtag_id, user_id, access_token, max_posts):
    """Fetch up to max_posts using pagination."""
    url = (f"https://graph.facebook.com/v22.0/{hashtag_id}/top_media?"
           f"fields=caption,like_count,media_url,media_product_type,permalink,id&"
           f"user_id={user_id}&limit={BATCH_SIZE}&access_token={access_token}")
    
    posts = []
    total_fetched = 0

    while url and total_fetched < max_posts:
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error during request: {e}")
            break

        batch = data.get("data", [])
        if not batch:
            print("No more posts found.")
            break

        posts.extend(batch[:max_posts - total_fetched])
        total_fetched += len(batch)

        print(f"Fetched {len(batch)} new posts (Total: {total_fetched}/{max_posts})")

        # Check for pagination
        url = data.get("paging", {}).get("next")
        time.sleep(1)  # Prevent hitting rate limits

    return posts

# Load existing data if file exists
if os.path.exists(JSON_FILENAME):
    with open(JSON_FILENAME, "r", encoding="utf-8") as jsonfile:
        try:
            existing_data = json.load(jsonfile)
        except json.JSONDecodeError:
            existing_data = {"data": []}
else:
    existing_data = {"data": []}

# --- MAIN SCRIPT ---
for hashtag in hashtags:
    hashtag = hashtag.strip()
    if not hashtag:
        continue

    print(f"\nProcessing #{hashtag}...")

    # Get Hashtag ID
    hashtag_id = get_hashtag_id(hashtag, USER_ID, ACCESS_TOKEN)
    if not hashtag_id:
        print(f"Skipping #{hashtag} (ID not found).")
        continue

    print(f"Fetched Hashtag ID for #{hashtag}: {hashtag_id}")

    # Fetch posts with pagination
    posts = fetch_posts(hashtag_id, USER_ID, ACCESS_TOKEN, POSTS_PER_HASHTAG)
    if not posts:
        print(f"No posts found for #{hashtag}.")
        continue

    # Append fetched data
    existing_data["data"].extend(posts)
    print(f"Added {len(posts)} records for #{hashtag}.")

    # Wait to avoid rate limits
    time.sleep(2)

# Save all data to JSON
with open(JSON_FILENAME, "w", encoding="utf-8") as jsonfile:
    json.dump(existing_data, jsonfile, indent=4)

print(f"\nData collection complete. All records saved in '{JSON_FILENAME}'.")



Processing #indianskincare...
Fetched Hashtag ID for #indianskincare: 17841545905094640
Fetched 50 new posts (Total: 50/10000)
Fetched 50 new posts (Total: 100/10000)
Fetched 50 new posts (Total: 150/10000)
Fetched 50 new posts (Total: 200/10000)
Fetched 50 new posts (Total: 250/10000)


KeyboardInterrupt: 

In [4]:
JSON_FILENAME="beauty.json"

### 📌 Now we will print our dataset collected 📝


In [5]:

import pandas as pd
import json

# Load JSON file
with open(JSON_FILENAME, "r", encoding="utf-8") as file:
    data = json.load(file)

# Normalize the JSON data
df = pd.json_normalize(data["data"])  # Extracts data inside "data" key

# Select required columns
df = df[["caption", "like_count", "media_url", "media_product_type", "permalink", "id"]]

# Save the structured data to CSV
# df.to_csv("output.csv", index=False)
df

,caption,like_count,media_url,media_product_type,permalink,id
0,Episode - 97 of Meme Xpress Unknown Facts...\n...,1364.0,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/DFAtJJHzD67/,18277799980300139
1,Vineeta Singh An IIM Alumus had declined Rs 1 ...,4358.0,https://scontent.cdninstagram.com/v/t51.2885-1...,FEED,https://www.instagram.com/p/Cb_sr2-uJ8U/,17956447813676941
2,Adding iPhone to this list . Raise your hands ...,1197.0,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/CY6al7iLh9W/,18189775927159145
3,winter am routine❄️\n\ncetaphil_india gentle c...,879.0,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/Cma6JKfhNmS/,17952732260181270
4,A few other things to note -\n\n1. Damp skin m...,22831.0,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/CjDx2EkrNfz/,17991528778513919
...,...,...,...,...,...,...
7290,Delivering now: Your at-home facial ritual to ...,658.0,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/CkBgZeoMTAC/,17987822947608958
7291,Is that a scoop of ice cream or Red Vine Face ...,347.0,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/Ci2yF0BsUZY/,17976692557732310
7292,Get the ultimate sun protection every time you...,471.0,https://scontent.cdninstagram.com/v/t51.2885-1...,FEED,https://www.instagram.com/p/Cj20FRrvSMf/,17946422726507448
7293,Clinkin’ cuz the weekend is here! 🥂\n.\nGet fe...,920.0,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/CUfKavjMRgW/,17913898388051859


In [ ]:
i want you update the code for taking the permalink column and like_count as in the image shown from the file named:beauty.json by loading in normalized form and run the script for fetch username and append the data in fileindianskincare.csv file with column username,(like_count cloumn ) with  appended suffixes like/view according to the permalink type you detect by your self by logic of tv/p/reel etc and correspond permalink column used ,update in the code but not change the xpath for finding username:

### 📌 Now we wil remove duplicate posts having same media id📝

In [9]:

# Display the first few rows
df
column_name = 'id'  # Change this to any column you want to check

# Count total duplicate values (excluding unique ones)
total_duplicates = df[column_name].duplicated().sum()

print(f"Total duplicate values in column '{column_name}': {total_duplicates}")

Total duplicate values in column 'id': 0


In [10]:
with open(JSON_FILENAME, "r", encoding="utf-8") as file:
    data = json.load(file)

# Convert JSON to DataFrame
df = pd.DataFrame(data["data"])

if df.empty:
    print("No data found in the JSON file.")
else:
    # Count total duplicate values before removal
    total_duplicates = df["id"].duplicated().sum()
    print(f"Total duplicate values in column 'id' before removal: {total_duplicates}")

    # Remove duplicate rows based on the 'id' column
    df = df.drop_duplicates(subset="id")

    # Reset row indices
    df = df.reset_index(drop=True)

    # Save cleaned data back to the same JSON file
    cleaned_data = df.to_dict(orient="records")
    with open(JSON_FILENAME, "w", encoding="utf-8") as file:
        json.dump({"data": cleaned_data}, file, indent=4)

    print(f"Duplicates removed. Data saved back to '{JSON_FILENAME}'.\n")
    print(df.head())

    # Display the first few rows after cleaning

Total duplicate values in column 'id' before removal: 0
Duplicates removed. Data saved back to 'beauty.json'.

                                             caption  \
0  Day 3/365 #ReStartwGur \nBaddie Back in the Be...   
1  🩵 glam room selfies \n.\n.\n.\n.\n#bts #blueli...   
2  Elegance of this beautiful hand painted organz...   
3  ✨GIVEAWAY ALERT✨ CLOSED\n\nI have got the DREA...   
4                                                  💚   

                                           media_url media_product_type  \
0  https://scontent.cdninstagram.com/v/t51.29350-...               FEED   
1  https://scontent.cdninstagram.com/v/t51.29350-...               FEED   
2  https://scontent.cdninstagram.com/v/t51.29350-...               FEED   
3  https://scontent.cdninstagram.com/v/t51.29350-...               FEED   
4  https://scontent.cdninstagram.com/v/t51.29350-...               FEED   

                                  permalink                 id  like_count  
0  https://www.instagram

In [11]:
print("total unique records:", len(df))

total unique records: 555


In [12]:
df

,caption,media_url,media_product_type,permalink,id,like_count
0,Day 3/365 #ReStartwGur \nBaddie Back in the Be...,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/C3KIeiPu6a7/,17951805287635639,NaN
1,🩵 glam room selfies \n.\n.\n.\n.\n#bts #blueli...,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/CuCLF2ep8hE/,17972038775393749,563.0
2,Elegance of this beautiful hand painted organz...,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/Cq5sQkgIYF-/,18005653312718386,8085.0
3,✨GIVEAWAY ALERT✨ CLOSED\n\nI have got the DREA...,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/CmyfjbhpLzT/,17902370273641548,2196.0
4,💚,https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/Cr9_9HWS9YJ/,18192248482266050,293.0
...,...,...,...,...,...,...
550,"Pyaara gaanaa, Pyaari Main. ❤️✨ \n\nShubho Mah...",https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/CyXbfWbJdsS/,17910141599765357,3289.0
551,"In these 4 walls, my thoughts are safe and I a...",https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/ClT1-OES7OL/,17964151456916597,2446.0
552,See this tip change your skin care game! ✨\n\n...,https://instagram.fdel11-2.fna.fbcdn.net/o1/v/...,FEED,https://www.instagram.com/tv/CcSGm9Chms1/,17972613301548764,662.0
553,"""FRESH beauty secrets!"" 🌿\n\nInspired by the c...",https://scontent.cdninstagram.com/v/t51.29350-...,FEED,https://www.instagram.com/p/ChZmYbstJXa/,18046736068355198,506.0


### 📌 Now we will print permalinks and extract shortcodes 📝

In [13]:
df["permalink"]

0       https://www.instagram.com/p/C3KIeiPu6a7/
1       https://www.instagram.com/p/CuCLF2ep8hE/
2       https://www.instagram.com/p/Cq5sQkgIYF-/
3       https://www.instagram.com/p/CmyfjbhpLzT/
4       https://www.instagram.com/p/Cr9_9HWS9YJ/
                         ...                    
550     https://www.instagram.com/p/CyXbfWbJdsS/
551     https://www.instagram.com/p/ClT1-OES7OL/
552    https://www.instagram.com/tv/CcSGm9Chms1/
553     https://www.instagram.com/p/ChZmYbstJXa/
554     https://www.instagram.com/p/ClA1TPphgDB/
Name: permalink, Length: 555, dtype: object

In [14]:
# Extract shortcodes from df["permalink"]
shortcodes = df["permalink"].apply(lambda x: x.split("/")[-2].split("?")[0])

# Print the shortcodes series
print(shortcodes)

# Check if the series is empty
if shortcodes.empty:
    print("The shortcodes series is empty.")
else:
    # Create a new DataFrame with the shortcodes
    shortcodes_df = pd.DataFrame(shortcodes, columns=["shortcode"])
    print(shortcodes_df)

0      C3KIeiPu6a7
1      CuCLF2ep8hE
2      Cq5sQkgIYF-
3      CmyfjbhpLzT
4      Cr9_9HWS9YJ
          ...     
550    CyXbfWbJdsS
551    ClT1-OES7OL
552    CcSGm9Chms1
553    ChZmYbstJXa
554    ClA1TPphgDB
Name: permalink, Length: 555, dtype: object
Empty DataFrame
Columns: [shortcode]
Index: []


In [15]:
print(type(shortcodes))
print(shortcodes)

<class 'pandas.core.series.Series'>
0      C3KIeiPu6a7
1      CuCLF2ep8hE
2      Cq5sQkgIYF-
3      CmyfjbhpLzT
4      Cr9_9HWS9YJ
          ...     
550    CyXbfWbJdsS
551    ClT1-OES7OL
552    CcSGm9Chms1
553    ChZmYbstJXa
554    ClA1TPphgDB
Name: permalink, Length: 555, dtype: object


### 📌 Now we will extract the usernames of the posts  📝

In [18]:
import time
import random
import pandas as pd
from instaloader import Instaloader, Post

# Initialize Instaloader
L = Instaloader()

# Load shortcodes from a CSV file (if needed)
# shortcodes = ["CximKgfq-Gu", "Ci98q31LR2i", "DBBX9kUzFXf", "DCI-p6Xv_OA"]  # Replace with your shortcode list

# Output file name
output_file = "output2.csv"

# Load existing data to avoid duplicates
try:
    existing_data = pd.read_csv(output_file)
    processed_shortcodes = set(existing_data["shortcodes"])
except FileNotFoundError:
    processed_shortcodes = set()

# Create an empty list to store results
results = []

# Initialize retry settings
max_retries = 5
initial_wait_time = 30  # Initial wait time in seconds for backoff

# Iterate over each shortcode
for shortcode in shortcodes:
    if shortcode in processed_shortcodes:
        print(f"Skipping {shortcode}, already processed.")
        continue  # Skip already processed shortcodes

    retries = 0
    wait_time = initial_wait_time

    while retries < max_retries:
        try:
            # Fetch the post details using the shortcode
            post = Post.from_shortcode(L.context, shortcode)

            # Get the owner's profile
            owner_profile = post.profile
            print(f"Fetched: {owner_profile}")

            # Append the result to the list
            results.append({
                "shortcodes": shortcode,
                "username": owner_profile  # Extract username
            })

            # Save the result to a CSV file efficiently
            pd.DataFrame(results[-1:], columns=["shortcode", "owner_profile"]).to_csv(
                output_file, mode='a', header=not processed_shortcodes, index=False
            )

            # Add to processed set
            processed_shortcodes.add(shortcode)

            # Randomized delay to avoid rate limits (3 to 8 sec)
            time.sleep(random.uniform(3, 8))
            break  # Move to next shortcode

        except Exception as e:
            error_msg = str(e)
            print(f"Error processing {shortcode}: {error_msg}")

            # If 401 Unauthorized, wait and retry
            if "401 Unauthorized" in error_msg or "Please wait a few minutes" in error_msg:
                print(f"Rate limit hit! Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
                wait_time *= 2  # Exponential backoff
                retries += 1
            else:
                # If another error, move to the next shortcode
                break

print("✅ Process completed successfully!")


Skipping C3KIeiPu6a7, already processed.
Skipping CuCLF2ep8hE, already processed.
Skipping Cq5sQkgIYF-, already processed.
Skipping CmyfjbhpLzT, already processed.
Skipping Cr9_9HWS9YJ, already processed.


JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query [retrying; skip with ^C]
JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?variables=%7B%22shortcode%22%3A%22Ckx7moqIb-s%22%7D&doc_id=8845758582119845&server_timestamps=true [retrying; skip with ^C]


Error processing Ckx7moqIb-s: JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?variables=%7B%22shortcode%22%3A%22Ckx7moqIb-s%22%7D&doc_id=8845758582119845&server_timestamps=true
Rate limit hit! Retrying in 30 seconds...


JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query [retrying; skip with ^C]
JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?variables=%7B%22shortcode%22%3A%22Ckx7moqIb-s%22%7D&doc_id=8845758582119845&server_timestamps=true [retrying; skip with ^C]


Error processing Ckx7moqIb-s: JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?variables=%7B%22shortcode%22%3A%22Ckx7moqIb-s%22%7D&doc_id=8845758582119845&server_timestamps=true
Rate limit hit! Retrying in 60 seconds...


...

JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query [retrying; skip with ^C]
JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?variables=%7B%22shortcode%22%3A%22CwcwHNZKqT-%22%7D&doc_id=8845758582119845&server_timestamps=true [retrying; skip with ^C]


Error processing CwcwHNZKqT-: JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?variables=%7B%22shortcode%22%3A%22CwcwHNZKqT-%22%7D&doc_id=8845758582119845&server_timestamps=true
Rate limit hit! Retrying in 240 seconds...


JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query [retrying; skip with ^C]
JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?variables=%7B%22shortcode%22%3A%22CwcwHNZKqT-%22%7D&doc_id=8845758582119845&server_timestamps=true [retrying; skip with ^C]


Error processing CwcwHNZKqT-: JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?variables=%7B%22shortcode%22%3A%22CwcwHNZKqT-%22%7D&doc_id=8845758582119845&server_timestamps=true
Rate limit hit! Retrying in 480 seconds...


KeyboardInterrupt: 

---
## 🔍 Now we will check for unique usernames  📊
---


In [ ]:
import pandas as pd

# Load CSV file
# Change this to your actual file path
df = pd.read_csv(output_file)

# Check for duplicate usernames
duplicates = df.duplicated(subset=['usernames'])  # Check for duplicates in 'username' column
total_duplicates = duplicates.sum()
print(f"Total duplicate usernames found: {total_duplicates}")

# Remove duplicates
df_cleaned = df.drop_duplicates(subset=['usernames'], keep='first')

# Print total records after removing duplicates
print(f"Total records after removing duplicates: {len(df_cleaned)}")

# Save the cleaned data back to the same file
df_cleaned.to_csv(file_path, index=False)

print("Duplicates removed and file updated successfully.")


In [ ]:
import pandas as pd
data_total=pd.read_csv("output.csv")
print(len(data_total))

In [30]:
# import pandas as pd

# # Your list of hashtags
# hashtags = ['#skincareindia', '#naturalskincareindia', '#skincareroutineindia', '#skincaretipsindia', '#indianskincareblogger',
#             '#glowingskinindia', '#skincareproductsindia', '#ayurvedaskinindia','#skincareproductsindia', '#GlowingSkinindia', '#SkincareRoutineidia', '#sunscreenindian', 
#             '#sunscreeneverydayindia', '#skindetanindia', '#indianskincare', '#pigmentationindia', 
#             '#darkspotsindia', '#sunscreenindia']
# # Remove the '#' symbol from each hashtag
# cleaned_hashtags = [hashtag.lstrip('#') for hashtag in hashtags]

# # Create a DataFrame with the cleaned hashtags
# df = pd.DataFrame(cleaned_hashtags, columns=['hashtagslist'])

# # Save to CSV
# df.to_csv('skincarehashtags.csv', index=False)

# print("CSV file has been saved successfully.")


CSV file has been saved successfully.


In [32]:
# d=pd.read_csv("skincarehashtags.csv")
# d

,hashtagslist
0,skincareindia
1,naturalskincareindia
2,skincareroutineindia
3,skincaretipsindia
4,indianskincareblogger
5,glowingskinindia
6,skincareproductsindia
7,ayurvedaskinindia
8,skincareproductsindia
9,GlowingSkinindia


In [ ]:
# import pandas as pd
# import csv
# import time
# from instagrapi import Client
# from requests.exceptions import RequestException

# # Instagram credentials
# USERNAME = "aakash_sharma413"
# PASSWORD = "@_&#+(@&@Aakash"  # Replace with your actual password

# # Initialize the Client
# cl = Client()

# # Login to Instagram
# try:
#     cl.login(USERNAME, PASSWORD)
#     print("✅ Login successful!")
# except Exception as e:
#     print(f"❌ Login failed: {e}")
#     exit()  # Stop execution if login fails

# # Read the CSV file containing usernames and hashtags
# df = pd.read_csv('finalusername.csv')

# # Open an output CSV file to write results
# output_file = 'updated_followers2s.csv'

# # Check if the file already exists to decide whether to write a header
# write_header = not pd.io.common.file_exists(output_file)

# with open(output_file, 'a', newline='') as file:
#     csv_writer = csv.writer(file)

#     # Write header if the file is new
#     if write_header:
#         csv_writer.writerow(['Username', 'Hashtag', 'Followers'])

#     # Process data in batches of 5 users to reduce waiting time
#     batch_size = 5
#     for i in range(0, len(df), batch_size):
#         batch = df.iloc[i:i + batch_size]  # Extract a batch of 5 users

#         for index, row in batch.iterrows():
#             usrname = row['Username']
#             hashtag = row['Hashtag']

#             try:
#                 # Fetch followers count using instagrapi
#                 user_id = cl.user_id_from_username(usrname)
#                 user_info = cl.user_info(user_id)
#                 followers = user_info.follower_count

#                 # Print and store data
#                 print(f"Username: {usrname}, Hashtag: {hashtag}, Followers: {followers}")
#                 csv_writer.writerow([usrname, hashtag, followers])

#             except (RequestException, ValueError) as e:
#                 print(f"❌ Error fetching data for {usrname}: {e}")
#                 print(f"⏳ Retrying once...")
#                 time.sleep(5)  # Wait for a bit before retrying
#                 try:
#                     # Retry fetching followers count after the wait
#                     user_id = cl.user_id_from_username(usrname)
#                     user_info = cl.user_info(user_id)
#                     followers = user_info.follower_count

#                     # Print and store data after retry
#                     print(f"Username: {usrname}, Hashtag: {hashtag}, Followers: {followers}")
#                     csv_writer.writerow([usrname, hashtag, followers])

#                 except (RequestException, ValueError) as e:
#                     print(f"❌ Failed again for {usrname}: {e}")
#                     print(f"Skipping {usrname}...")

#         # Reduce waiting time by processing in batches
#         print("⏳ Waiting 5 seconds before processing the next batch...")
#         time.sleep(5)



In [6]:
# #run from next rows
# import pandas as pd
# import csv
# import time
# from instagrapi import Client

# # Instagram credentials
# USERNAME = "aakash_sharma413"
# PASSWORD = "@_&#+(@&@Aakash"  # Replace with your actual password

# # Initialize the Client
# cl = Client()

# # Login to Instagram
# try:
#     cl.login(USERNAME, PASSWORD)
#     print("✅ Login successful!")
# except Exception as e:
#     print(f"❌ Login failed: {e}")
#     exit()  # Stop execution if login fails

# # Read the CSV file containing usernames and hashtags
# df = pd.read_csv('finalusername.csv')

# # Open an output CSV file to write results
# output_file = 'updated_followers2s.csv'

# # Check if the file already exists to decide whether to write a header
# write_header = not pd.io.common.file_exists(output_file)

# with open(output_file, 'a', newline='') as file:
#     csv_writer = csv.writer(file)

#     # Write header if the file is new
#     if write_header:
#         csv_writer.writerow(['Username', 'Hashtag', 'Followers'])

#     # Start processing from the next row after the last processed user
#     start_index = df[df['Username'] == 'al_zenaskincarelasem'].index[0] + 1  # Get the index of the next row

#     # Process the data from the next user onward
#     for index, row in df.iloc[start_index:].iterrows():
#         usrname = row['Username']
#         hashtag = row['Hashtag']

#         try:
#             # Fetch followers count using instagrapi
#             user_id = cl.user_id_from_username(usrname)
#             user_info = cl.user_info(user_id)
#             followers = user_info.follower_count

#             # Print and store data
#             print(f"Username: {usrname}, Hashtag: {hashtag}, Followers: {followers}")
#             csv_writer.writerow([usrname, hashtag, followers])

#         except Exception as e:
#             print(f"❌ Error fetching data for {usrname}: {e}")
        
#         # Optional: If you want to limit requests per minute, you can add a sleep time here, but for now, no delay


✅ Login successful!


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/110_avenue_aubagne/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/110_avenue_aubagne/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/110_avenue_aubagne/?__a=1&__d=dis) >>> 


Username: 110_avenue_aubagne, Hashtag: skincareroutine, Followers: 75


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/thammachad_official/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/thammachad_official/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/thammachad_official/?__a=1&__d=dis) >>> 


Username: thammachad_official, Hashtag: skincareroutine, Followers: 19


...

Username: infotronicsweb, Hashtag: indianskincareroutine, Followers: 3


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/_viju_s/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/_viju_s/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/_viju_s/?__a=1&__d=dis) >>> 


Username: _viju_s, Hashtag: indianskincareroutine, Followers: 2340


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/dreamyatoms/?__a=1&__d=dis) >>> 


KeyboardInterrupt: 

In [1]:
import instaloader

L = instaloader.Instaloader()
username = "myplixlife"

# Login with an Instagram account (optional but recommended)
L.login("aakash_sharma_413", "@1234Aakash")

# Get tagged posts
profile = instaloader.Profile.from_username(L.context, username)
for post in profile.get_tagged_posts():
    print(post.url)  # Print the post URL


JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?query_hash=e31a871f7301132ceaab56507a66bbb7&variables=%7B%22id%22%3A8628456936%2C%22first%22%3A12%7D [retrying; skip with ^C]
JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?query_hash=e31a871f7301132ceaab56507a66bbb7&variables=%7B%22id%22%3A8628456936%2C%22first%22%3A12%7D [retrying; skip with ^C]


ConnectionException: JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?query_hash=e31a871f7301132ceaab56507a66bbb7&variables=%7B%22id%22%3A8628456936%2C%22first%22%3A12%7D

In [5]:
import instaloader
import time

# Initialize Instaloader
L = instaloader.Instaloader()

# Instagram Username to Check
username = "myplixlife"

# Load saved session instead of logging in with a password
try:
    L.load_session_from_file("aakash_sharma_413")  # Load existing session
    print("✅ Session loaded successfully!")
except FileNotFoundError:
    print("⚠ Session file not found. Please log in manually first.")
    L.login("aakash_sharma_413", "@1234Aakash")  # One-time manual login
    L.save_session_to_file()  # Save session for future use

# Get tagged posts
try:
    profile = instaloader.Profile.from_username(L.context, username)
    print(f"🔍 Fetching tagged posts for {username}...\n")

    # Loop through tagged posts with rate-limiting
    for post in profile.get_tagged_posts():
        print(post.url)
        time.sleep(10)  # Pause for 10 seconds between requests to avoid blocking

    print("\n✅ Done fetching all tagged posts!")

except instaloader.exceptions.ProfileNotExistsException:
    print("❌ Error: The specified Instagram profile does not exist.")
except instaloader.exceptions.ConnectionException:
    print("❌ Error: Connection issue. Try again later.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")


Loaded session from C:\Users\91954\AppData\Local\Instaloader\session-aakash_sharma_413.
✅ Session loaded successfully!
🔍 Fetching tagged posts for myplixlife...



JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?query_hash=e31a871f7301132ceaab56507a66bbb7&variables=%7B%22id%22%3A8628456936%2C%22first%22%3A12%7D [retrying; skip with ^C]
JSON Query to graphql/query: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://www.instagram.com/graphql/query?query_hash=e31a871f7301132ceaab56507a66bbb7&variables=%7B%22id%22%3A8628456936%2C%22first%22%3A12%7D [retrying; skip with ^C]


❌ Error: Connection issue. Try again later.


In [7]:
import requests
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Instagram credentials (use your own)
INSTAGRAM_USERNAME = "aakash_sharma_413"
INSTAGRAM_PASSWORD = "@1234Aakash"
USERNAME_TO_FETCH = "myplixlife"  # Target username

# 🔹 Step 1: Log in to Instagram & Get Session ID
def get_instagram_session():
    options = webdriver.ChromeOptions()
    # options.add_argument("--headless")  # Run in headless mode (no UI)
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    driver.get("https://www.instagram.com/accounts/login/")
    time.sleep(5)  # Allow page to load

    # Enter username & password
    driver.find_element(By.NAME, "username").send_keys(INSTAGRAM_USERNAME)
    driver.find_element(By.NAME, "password").send_keys(INSTAGRAM_PASSWORD + Keys.RETURN)
    time.sleep(5)  # Wait for login to complete

    # Get cookies
    cookies = driver.get_cookies()
    driver.quit()

    # Extract sessionid
    sessionid = next((cookie["value"] for cookie in cookies if cookie["name"] == "sessionid"), None)
    if not sessionid:
        raise Exception("❌ Failed to retrieve sessionid!")
    
    print(f"✅ Session ID retrieved: {sessionid}")
    return sessionid

# 🔹 Step 2: Fetch Tagged Posts Using GraphQL
def get_tagged_posts(sessionid):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Cookie": f"sessionid={sessionid};"
    }

    # Get user ID
    user_info_url = f"https://www.instagram.com/{USERNAME_TO_FETCH}/?__a=1&__d=dis"
    response = requests.get(user_info_url, headers=headers)
    if response.status_code != 200:
        print("❌ Error: Failed to fetch user ID. Check session ID.")
        return

    user_id = response.json()["graphql"]["user"]["id"]

    # Fetch tagged posts
    query_hash = "e31a871f7301132ceaab56507a66bbb7"
    variables = json.dumps({"id": user_id, "first": 10})  # Fetch first 10 posts
    graphql_url = f"https://www.instagram.com/graphql/query?query_hash={query_hash}&variables={variables}"

    response = requests.get(graphql_url, headers=headers)
    if response.status_code != 200:
        print("❌ Error: GraphQL request failed. Instagram might be blocking requests.")
        return

    data = response.json()
    tagged_posts = data["data"]["user"]["edge_user_to_photos_of_you"]["edges"]

    # Print post URLs
    for post in tagged_posts:
        post_node = post["node"]
        print(f"📸 Post: https://www.instagram.com/p/{post_node['shortcode']}/")
        time.sleep(5)  # Delay to avoid rate-limiting

# 🔹 Execute the Functions
session_id = get_instagram_session()
get_tagged_posts(session_id)


✅ Session ID retrieved: 71968213252%3AqBVBnugaavTa3r%3A23%3AAYdaDd1RC2vSfznu-bdTKPWRg13HOPQCsU74NWDsyQ
❌ Error: Failed to fetch user ID. Check session ID.


In [8]:
import requests
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Instagram credentials (use your own)
INSTAGRAM_USERNAME = "aakash_sharma_413"
INSTAGRAM_PASSWORD = "@1234Aakash"
USERNAME_TO_FETCH = "myplixlife"  # Target username

SESSION_FILE = "sessionid.txt"  # File to store session ID

# 🔹 Step 1: Get Session ID
def get_instagram_session():
    try:
        # Load session ID if it exists
        with open(SESSION_FILE, "r") as f:
            sessionid = f.read().strip()
            if sessionid:
                print(f"✅ Using saved session ID: {sessionid}")
                return sessionid
    except FileNotFoundError:
        pass  # No session file, continue login

    # Start Selenium WebDriver (headless mode)
    options = webdriver.ChromeOptions()
    # options.add_argument("--headless")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    driver.get("https://www.instagram.com/accounts/login/")
    time.sleep(5)  # Wait for page to load

    # Enter username & password
    driver.find_element(By.NAME, "username").send_keys(INSTAGRAM_USERNAME)
    driver.find_element(By.NAME, "password").send_keys(INSTAGRAM_PASSWORD + Keys.RETURN)
    time.sleep(5)  # Wait for login to complete

    # Get session ID from cookies
    cookies = driver.get_cookies()
    driver.quit()
    sessionid = next((cookie["value"] for cookie in cookies if cookie["name"] == "sessionid"), None)
    
    if not sessionid:
        raise Exception("❌ Failed to retrieve sessionid!")

    # Save session ID for future use
    with open(SESSION_FILE, "w") as f:
        f.write(sessionid)

    print(f"✅ Session ID retrieved and saved: {sessionid}")
    return sessionid

# 🔹 Step 2: Fetch Tagged Posts Using GraphQL
def get_tagged_posts(sessionid):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Cookie": f"sessionid={sessionid};"
    }

    # Get user ID
    user_info_url = f"https://www.instagram.com/{USERNAME_TO_FETCH}/?__a=1&__d=dis"
    response = requests.get(user_info_url, headers=headers)

    if response.status_code != 200:
        print(f"❌ Error: Failed to fetch user ID. Status Code: {response.status_code}")
        print("🔍 Response:", response.text)
        return

    user_data = response.json()
    user_id = user_data.get("graphql", {}).get("user", {}).get("id")

    if not user_id:
        print("❌ Error: Unable to extract user ID. Instagram may have changed the API.")
        return

    # Fetch tagged posts
    query_hash = "e31a871f7301132ceaab56507a66bbb7"
    variables = json.dumps({"id": user_id, "first": 10})  # Fetch first 10 posts
    graphql_url = f"https://www.instagram.com/graphql/query?query_hash={query_hash}&variables={variables}"

    response = requests.get(graphql_url, headers=headers)
    if response.status_code != 200:
        print(f"❌ Error: GraphQL request failed. Status Code: {response.status_code}")
        print("🔍 Response:", response.text)
        return

    data = response.json()
    tagged_posts = data.get("data", {}).get("user", {}).get("edge_user_to_photos_of_you", {}).get("edges", [])

    # Print post URLs
    for post in tagged_posts:
        post_node = post["node"]
        print(f"📸 Post: https://www.instagram.com/p/{post_node['shortcode']}/")
        time.sleep(5)  # Delay to avoid rate-limiting

# 🔹 Execute the Functions
session_id = get_instagram_session()
get_tagged_posts(session_id)


✅ Session ID retrieved and saved: 71968213252%3Aq5p8iJEHj5yTWP%3A17%3AAYeNpoq_tu4Jf3YR8WrHOKiSOzr4Y2zvXlTFk5sq5w
❌ Error: Failed to fetch user ID. Status Code: 201
🔍 Response: 


In [2]:
import instaloader

L = instaloader.Instaloader()
L.load_session_from_file("aakash_sharma_413")  # Load saved session

username = "myplixlife"
profile = instaloader.Profile.from_username(L.context, username)

for post in profile.get_tagged_posts():
    print(post.url)


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\users\\91954\\appdata\\local\\temp/.instaloader-91954/session-aakash_sharma_413'